# 멀티모달 LLM - 실습 코드 2: CLIP 임베딩으로 이미지-텍스트 유사도 계산

- Tutorial ID: `expand-multimodal-llm`
- Tutorial: 멀티모달 LLM
- Section ID: `expand-multimodal-llm-code-2`
- Section: 실습 코드 2: CLIP 임베딩으로 이미지-텍스트 유사도 계산


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: CLIP 임베딩으로 이미지-텍스트 유사도 계산
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
import requests

# CLIP 모델 로드
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# ── 이미지-텍스트 매칭 ──
# 샘플 이미지 로드
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# 여러 텍스트 후보
texts = [
    "a photo of two cats sleeping on a couch",
    "a photo of a dog running in the park", 
    "a picture of cars on the street",
    "an image of people eating dinner",
]

# 이미지-텍스트 유사도 계산
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = model(**inputs)
        logits = outputs.logits_per_image  # (1, num_texts)
        probs = logits.softmax(dim=1)

print("=== CLIP 이미지-텍스트 유사도 ===")
print(f"이미지: 두 마리 고양이가 소파에서 자는 사진\n")
for text, prob in zip(texts, probs[0]):
    bar = "█" * int(prob * 50)
    print(f"  {prob:.3f} {bar} \"{text}\"")

# ── Zero-shot 이미지 분류 ──
print("\n=== Zero-shot 분류 ===")
cifar10_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
class_prompts = [f"a photo of a {c}" for c in cifar10_classes]

inputs = processor(text=class_prompts, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)[0]

top5 = torch.topk(probs, 5)
print("Top-5 예측:")
for i, (score, idx) in enumerate(zip(top5.values, top5.indices)):
    print(f"  {i+1}. {cifar10_classes[idx]} ({score:.3f})")